# Phase 5 — RoBERTa Fine-tuning

Fine-tuning του `roberta-base` για το Food Hazard Detection task.

**Βελτιώσεις σε σχέση με το DistilBERT notebook:**
- Χρησιμοποιούμε **train+valid μαζί** για εκπαίδευση → αποφεύγουμε το overfitting στο validation
- **Σταθερά 5 epochs** αντί για early stopping που προκαλούσε data leakage
- `roberta-base` αντί `distilbert` → καλύτερο μοντέλο
- Αποθήκευση probabilities για ensemble (Βήμα 3)

**Γιατί train+valid μαζί:**
Στο DistilBERT χρησιμοποιήσαμε το validation για early stopping.
Αυτό σημαίνει ότι το μοντέλο 'είδε' έμμεσα τα validation labels
→ inflated validation score (0.7998) αλλά χαμηλό Kaggle score (0.723).
Τώρα εκπαιδεύουμε σε όλα τα δεδομένα και υποβάλλουμε απευθείας.

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Φόρτωση δεδομένων
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Συνδυάζουμε train + valid για εκπαίδευση
# Αυτό είναι το κλειδί: δεν κρατάμε validation για early stopping
train_full = pd.concat([train, valid], ignore_index=True)

print(f'Train only:       {len(train)} δείγματα')
print(f'Valid only:       {len(valid)} δείγματα')
print(f'Train+Valid:      {len(train_full)} δείγματα  ← αυτό χρησιμοποιούμε')
print(f'Test:             {len(test)} δείγματα')

In [ ]:
# Προετοιμασία κειμένων — ίδια λογική με Phase 1 (βέλτιστη)
def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train_full = make_text(train_full)
texts_test       = make_text(test)

# Κρατάμε το validation χωριστά ΜΟΝΟ για αξιολόγηση μετά
texts_valid = make_text(valid)

print(f'Train+Valid texts: {len(texts_train_full)}')
print(f'Test texts:        {len(texts_test)}')
print(f'\nΠαράδειγμα: {texts_train_full[0][:150]}...')

In [ ]:
# Label Encoding
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

# Fit στο train_full για να δει όλες τις κλάσεις
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_train_hazard  = le_hazard.transform(train_full['hazard-category'])
y_train_product = le_product.transform(train_full['product-category'])

# Valid labels για post-hoc αξιολόγηση
y_valid_hazard  = le_hazard.transform(valid['hazard-category'])
y_valid_product = le_product.transform(valid['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)

print(f'Hazard classes:  {n_hazard}  → {list(le_hazard.classes_)}')
print(f'Product classes: {n_product}')

In [ ]:
# Dataset class για RoBERTa
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# Φόρτωση RoBERTa tokenizer
MODEL_NAME = 'roberta-base'
print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer έτοιμος!')

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
MAX_LENGTH = 128
N_EPOCHS   = 5      # Σταθερά 5 epochs — χωρίς early stopping
LR         = 2e-5

# Datasets
train_dataset_hazard  = FoodHazardDataset(texts_train_full, y_train_hazard,  tokenizer, MAX_LENGTH)
train_dataset_product = FoodHazardDataset(texts_train_full, y_train_product, tokenizer, MAX_LENGTH)

# Valid dataset για post-hoc έλεγχο (δεν χρησιμοποιείται για early stopping)
valid_dataset_hazard  = FoodHazardDataset(texts_valid, y_valid_hazard,  tokenizer, MAX_LENGTH)
valid_dataset_product = FoodHazardDataset(texts_valid, y_valid_product, tokenizer, MAX_LENGTH)

# Test dataset (dummy labels)
dummy_labels = np.zeros(len(texts_test), dtype=int)
test_dataset_hazard  = FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH)
test_dataset_product = FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH)

# DataLoaders
train_loader_hazard  = DataLoader(train_dataset_hazard,  batch_size=BATCH_SIZE, shuffle=True)
train_loader_product = DataLoader(train_dataset_product, batch_size=BATCH_SIZE, shuffle=True)
valid_loader_hazard  = DataLoader(valid_dataset_hazard,  batch_size=BATCH_SIZE, shuffle=False)
valid_loader_product = DataLoader(valid_dataset_product, batch_size=BATCH_SIZE, shuffle=False)
test_loader_hazard   = DataLoader(test_dataset_hazard,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_product  = DataLoader(test_dataset_product,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches (hazard):  {len(train_loader_hazard)}')
print(f'Train batches (product): {len(train_loader_product)}')
print(f'Epochs: {N_EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LR}')

In [ ]:
# Helper functions
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product,
                       verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard,
                         average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    correct_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_mask.sum()
    if n_correct == 0:
        f1_product = 0.0
    else:
        f1_product = f1_score(
            y_true_product[correct_mask],
            y_pred_product[correct_mask],
            average='macro', zero_division=0
        )
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

## Εκπαίδευση RoBERTa — Hazard Classifier

Εκπαίδευση για 5 epochs στο **πλήρες train+valid set**.
Δεν χρησιμοποιούμε early stopping — αποφεύγουμε data leakage.

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ RoBERTa — HAZARD ===')
print(f'Δεδομένα: {len(train_full)} δείγματα (train+valid)')
print(f'Epochs: {N_EPOCHS}, LR: {LR}\n')

# Φόρτωση μοντέλου
roberta_hazard = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_hazard
).to(device)

optimizer_hazard = AdamW(roberta_hazard.parameters(), lr=LR)

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_hazard, train_loader_hazard, optimizer_hazard)

    # Post-hoc αξιολόγηση στο valid (ΔΕΝ επηρεάζει εκπαίδευση)
    preds_enc = get_predictions(roberta_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f} | Valid macro-F1 Hazard: {f1:.4f}')

# Αποθήκευση
torch.save(roberta_hazard.state_dict(), 'roberta_hazard_model.pt')
print('\nΑποθηκεύτηκε: roberta_hazard_model.pt')

## Εκπαίδευση RoBERTa — Product Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ RoBERTa — PRODUCT ===')
print(f'Δεδομένα: {len(train_full)} δείγματα (train+valid)')
print(f'Epochs: {N_EPOCHS}, LR: {LR}\n')

roberta_product = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_product
).to(device)

optimizer_product = AdamW(roberta_product.parameters(), lr=LR)

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_product, train_loader_product, optimizer_product)

    preds_enc = get_predictions(roberta_product, valid_loader_product)
    preds     = le_product.inverse_transform(preds_enc)
    f1        = f1_score(valid['product-category'], preds, average='macro', zero_division=0)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f} | Valid macro-F1 Product: {f1:.4f}')

torch.save(roberta_product.state_dict(), 'roberta_product_model.pt')
print('\nΑποθηκεύτηκε: roberta_product_model.pt')

## Αξιολόγηση στο Validation Set

Σημείωση: το validation set ήταν μέρος των training data,
οπότε αυτό το score είναι ενδεικτικό — το αληθινό score
θα φανεί στο Kaggle.

In [ ]:
# Predictions στο validation
preds_hazard_enc  = get_predictions(roberta_hazard,  valid_loader_hazard)
preds_product_enc = get_predictions(roberta_product, valid_loader_product)

preds_hazard_roberta  = le_hazard.inverse_transform(preds_hazard_enc)
preds_product_roberta = le_product.inverse_transform(preds_product_enc)

print('=== ΑΞΙΟΛΟΓΗΣΗ RoBERTa (train+valid) ===')
print('(Προσοχή: valid ήταν training data — score overestimated)\n')
official_st1_score(
    valid['hazard-category'], preds_hazard_roberta,
    valid['product-category'], preds_product_roberta
)

print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('TF-IDF + SVM:                    0.7419 (valid) / ~0.745 (Kaggle)')
print('DistilBERT fine-tuned (overfitting): 0.7998 (valid) / 0.723 (Kaggle)')
print('BERT+SVM Ensemble:               0.8028 (valid) / 0.755 (Kaggle)')

## Submission + Αποθήκευση Probabilities για Ensemble

Αποθηκεύουμε:
1. `submission_roberta.csv` — για απευθείας υποβολή στο Kaggle
2. `roberta_test_hazard_probs.npy` — για ensemble με DeBERTa + SVM (Βήμα 3)

In [ ]:
# Test predictions
preds_test_hazard_enc  = get_predictions(roberta_hazard,  test_loader_hazard)
preds_test_product_enc = get_predictions(roberta_product, test_loader_product)

preds_test_hazard  = le_hazard.inverse_transform(preds_test_hazard_enc)
preds_test_product = le_product.inverse_transform(preds_test_product_enc)

# Submission file
submission_roberta = pd.DataFrame({
    'id': test['id'],
    'hazard-category': preds_test_hazard,
    'product-category': preds_test_product
})
submission_roberta.to_csv('submission_roberta.csv', index=False)
print('Αποθηκεύτηκε: submission_roberta.csv')
print(f'Predictions: {len(submission_roberta)}')
print(submission_roberta.head())

In [ ]:
# Αποθήκευση probabilities για ensemble (Βήμα 3)
# Valid probabilities
roberta_valid_hazard_probs  = get_probabilities(roberta_hazard,  valid_loader_hazard)
roberta_valid_product_probs = get_probabilities(roberta_product, valid_loader_product)

# Test probabilities
roberta_test_hazard_probs  = get_probabilities(roberta_hazard,  test_loader_hazard)
roberta_test_product_probs = get_probabilities(roberta_product, test_loader_product)

np.save('roberta_valid_hazard_probs.npy',  roberta_valid_hazard_probs)
np.save('roberta_valid_product_probs.npy', roberta_valid_product_probs)
np.save('roberta_test_hazard_probs.npy',   roberta_test_hazard_probs)
np.save('roberta_test_product_probs.npy',  roberta_test_product_probs)

print(f'Valid hazard probs:   {roberta_valid_hazard_probs.shape}')
print(f'Valid product probs:  {roberta_valid_product_probs.shape}')
print(f'Test hazard probs:    {roberta_test_hazard_probs.shape}')
print(f'Test product probs:   {roberta_test_product_probs.shape}')
print('\nΌλα αποθηκεύτηκαν για ensemble!')

## Σύνοψη

| Αρχείο | Χρήση |
|---|---|
| `submission_roberta.csv` | Υποβολή στο Kaggle |
| `roberta_valid_hazard_probs.npy` | Ensemble (Βήμα 3) |
| `roberta_valid_product_probs.npy` | Ensemble (Βήμα 3) |
| `roberta_test_hazard_probs.npy` | Ensemble (Βήμα 3) |
| `roberta_test_product_probs.npy` | Ensemble (Βήμα 3) |
| `roberta_hazard_model.pt` | Αποθηκευμένο μοντέλο |
| `roberta_product_model.pt` | Αποθηκευμένο μοντέλο |

**Επόμενο βήμα:** Ίδιο notebook για DeBERTa-v3-small (Phase 6).